> 📄 English: [multiuser-agent-arcade.ipynb](multiuser-agent-arcade.ipynb)
![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade)

# 使用 Arcade.dev 实现安全的工具调用

本教程将帮你弥补演示级 Agent 与生产级 Agent 之间最大的差距之一：__安全的工具调用（Secure tool-calling）__

当你的 Agent 在你自己的电脑上运行良好时，它们是不错的个人助手；但要把它们扩展到多用户并不容易，因为本地部署的安全假设并不适用于规模化场景下的 Agent。对于多用户来说，个人访问令牌（Personal Access Tokens）根本不够用。即便你把所有功能都封装到一个远程 MCP 服务器中，工具级认证仍需要你为 Agent 所依赖的所有提供商实现各自的认证流程。

Arcade 通过为 Agent 工具执行提供一个统一平台来解决这个问题。它会替你处理认证流程，为你的 Agent 提供一个安全的多用户解决方案。

在本教程中，你将学习如何使用 Arcade 和 LangGraph：
- 构建 Agent
- 提供能与以下服务交互的工具
    - GMail
    - Slack
    - Notion
- 在调用特定工具时实现安全护栏（Human-in-the-Loop）

# 环境准备

在进入代码之前，先搭建好具备正确依赖的开发环境

---



## 系统流程
下面展示了如何从一个简单的聊天机器人，成长为一个具备完整安全控制能力的健壮多用户系统：


# <div align="center">
#   <img src="arcade-diagram.png" alt="Arcade Multi-User Secure Tool Calling Architecture" width="700"/>
# </div>


## 开发环境设置

在实现多用户 Agent 系统之前，我们需要先搭建一个具备必要依赖的开发环境。下面的安装包括：用于 Agent 编排的 LangGraph、用于工具集成的 LangChain-Arcade，以及支持 OpenAI 的核心 LangChain 库。


In [ ]:
!pip install langgraph langchain-arcade "langchain[openai]"

## API Key 配置

本教程需要两个关键的 API key 才能运行。你需要一个 [OpenAI API](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fsignup&text=OpenAI%20API) key，以及一个 [Arcade API](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade&click=arcade-signup&target=https%3A%2F%2Fapi.arcade.dev%2Fsignup%3Futm_source%3Dgithub%26utm_medium%3Dnotebook%26utm_campaign%3Dnir_diamant%26utm_content%3Dtutorial&text=Arcade%20API) key。这两个服务的注册流程都很简单，其中 Arcade 专为简化外部工具集成到 AI 应用而设计。


In [ ]:
import getpass
import os

def _set_env(key: str, default: str | None):
    if key not in os.environ:
        if default:
            os.environ[key] = default
        else:
            os.environ[key] = getpass.getpass(f"{key}:")


_set_env("OPENAI_API_KEY")
_set_env("ARCADE_API_KEY")

## 用户身份配置

Arcade 平台需要用户标识，以便正确管理工具授权并维护不同用户之间的安全边界。该标识必须与创建 Arcade 账户时使用的邮箱地址一致，从而确保工具权限和 OAuth token 正确关联到对应的用户账户。


In [ ]:
_set_env("ARCADE_USER_ID")

# 简单对话 Agent

我们从实现一个基础的对话 Agent 开始，它演示了不依赖外部工具的核心 LangGraph 功能。这个基础 Agent 具备带短期记忆的对话能力，能在整个对话过程中保持上下文，同时建立起我们在本教程后续将不断扩展的架构模式。

## 核心 Agent 实现

下面的实现使用 [LangGraph 和 Arcade](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade&click=arcade-docs&target=https%3A%2F%2Fdocs.arcade.dev%2Fhome%2Flangchain%2Fuse-arcade-tools%23create-a-react-style-agent%3Futm_source%3Dgithub%26utm_medium%3Dnotebook%26utm_campaign%3Dnir_diamant%26utm_content%3Dtutorial&text=LangGraph%20and%20Arcade) 创建了一个 React 风格的 Agent。我们通过 MemorySaver 检查点器（checkpointer）为其配置对话记忆，让 Agent 能记住同一对话线程中的先前交互。Agent 收到一个清晰的提示词，定义了其乐于助人、简洁的个性，以及处理含糊请求的指令。


In [ ]:
from langgraph.prebuilt.chat_agent_executor import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
import uuid

# create a checkpointer to persist the graph's state
checkpointer = MemorySaver()

agent_a = create_react_agent(
    model="openai:gpt-5",
    prompt="You are a helpful assistant that can help with everyday tasks."
           " If the user's request is confusing you must ask them to clarify"
           " their intent, and fulfill the instruction to the best of your"
           " ability. Be concise and friendly at all times.",
    tools=[], # no tools for now!
    checkpointer=checkpointer
)

## Agent 交互工具函数

为了便于在本教程中与我们的 Agent 进行一致的交互，我们实现了一个工具函数，用于流式输出 Agent 的响应并以可读格式显示。该函数处理图的流式输出，并展示每个交互周期的最新消息，从而在 Agent 对话期间提供即时反馈。


In [ ]:
from langgraph.graph.state import CompiledStateGraph
def run_graph(graph: CompiledStateGraph, config, input):
    for event in graph.stream(input, config=config, stream_mode="values"):
        if "messages" in event:
            event["messages"][-1].pretty_print()

## 交互式聊天界面

下面的实现为测试基础 Agent 提供了一个完整的交互式聊天界面。系统为每个会话生成唯一的对话线程标识，使记忆能在同一对话的多次交流中持久保存。用户可以自然地与 Agent 交流，输入 "exit" 即可结束会话。


In [ ]:
# the configuration helps LangGraph keep track of conversations and interrups
# While it's not needed for this agent. The agent will remember different
# conversations based on the thread_id. This code generates a random id every
# time you run the cell, but you can hardcode the thread_id if you want to
# test the memory.
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}
while True:
    user_input = input("👤: ")
    # let's use "exit" as a safe way to break the infinite loop
    if user_input.lower() == "exit":
        break

    user_message = {"messages": [HumanMessage(content=user_input)]}
    run_graph(agent_a, config, user_message)



## 测试 Agent 的局限

为了理解基础 Agent 的边界，我们用需要访问外部数据的请求来测试它。下面的测试展示了 Agent 无法提供当前日期信息，因为大多数语言模型缺乏实时数据访问能力，可能会给出过时或不准确的时间信息。


In [ ]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}
print(f'thread_id = {config["configurable"]["thread_id"]}')

prompt = "what's today's date?"
user_message = {"messages": [HumanMessage(content=prompt)]}
run_graph(agent_a, config, user_message)

## 演示认证需求

下面的测试说明了 Agent 完全无法访问私有的、需要认证的数据源。当被要求总结个人邮件时，如果没有适当的认证机制以及对外部服务的授权访问，Agent 就无法继续。这一局限凸显了在生产级 Agent 系统中进行安全工具集成的关键必要性。


In [ ]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}
print(f'thread_id = {config["configurable"]["thread_id"]}')

prompt = "summarize my latest 3 emails please"
user_message = {"messages": [HumanMessage(content=prompt)]}
run_graph(agent_a, config, user_message)

# 带安全认证的工具集成

在建立了基础 Agent 架构之后，我们现在来解决实现外部服务安全访问这一核心挑战。本节将演示 Arcade.dev 如何解决工具级认证这一复杂问题，提供一种可跨多用户、多服务扩展的简化 OAuth 集成方式。

## Arcade 客户端初始化

我们首先通过核心客户端和 LangChain 集成层建立与 Arcade 平台的连接。ToolManager 是我们配置和授权工具的主要接口，而 Arcade 客户端则负责底层的认证基础设施。


In [ ]:
from langchain_arcade import ToolManager
from arcadepy import Arcade

arcade_client = Arcade(api_key=os.getenv("ARCADE_API_KEY"))
manager = ToolManager(client=arcade_client)

## Gmail 工具配置

我们的第一个工具集成聚焦于 Gmail 访问，特别是基础 Agent 无法提供的邮件列表功能。Gmail_ListEmails 工具让 Agent 能够获取并分析邮件数据，但在访问私有邮件账户之前，需要用户进行适当的授权。


In [ ]:
gmail_tool = manager.init_tools(tools=["Gmail_ListEmails"])[0]

## 授权工具函数

为了简化本教程中的授权流程，我们实现了一个可复用的函数来处理 OAuth 流程的发起与完成。不过，要读取我们的邮件，需要以安全的方式授予应用读取权限。Arcade 通过[替我们处理 OAuth2](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade&click=arcade-docs&target=https%3A%2F%2Fdocs.arcade.dev%2Fhome%2Fauth%2Fhow-arcade-helps%3Futm_source%3Dgithub%26utm_medium%3Dnotebook%26utm_campaign%3Dnir_diamant%26utm_content%3Dtutorial&text=handling%20the%20OAuth2%20for%20us) 让这一切变得简单。该函数会检查特定工具与用户组合的当前授权状态，必要时发起 OAuth 流程，并等待用户完成授权。


In [ ]:
def authorize_tool(tool_name, user_id, manager):
    # This line will check if this user is authorized to use the
    # tool, and return a response that we can use if the user
    # did not authorize the tool yet.
    auth_response = manager.authorize(
        tool_name=tool_name,
        user_id=user_id
    )
    if auth_response.status != "completed":
        print(f"The app wants to use the {tool_name} tool.\n"
              f"Please click this url to authorize it {auth_response.url}")
        # wait until the user authorizes
        manager.wait_for_auth(auth_response.id)


## Gmail 授权流程

下面的单元格发起 Gmail 访问的授权流程。如果用户此前未授予权限，Arcade 会提供一个 OAuth URL 用于完成授权。一旦完成授权，该权限会在后续会话中持久保存，无需重复进行授权流程。


In [ ]:
authorize_tool(gmail_tool.name, os.getenv("ARCADE_USER_ID"), manager)

## 带 Gmail 能力的增强版 Agent

完成 Gmail 授权后，我们现在可以创建一个具备邮件访问能力的增强版 Agent。这个 Agent 保留了基础实现的所有对话能力，同时增加了与经过认证的邮件服务交互的能力。请注意更新后的提示词明确提到了 Gmail 能力，并且配置中加入了 user_id 用于工具执行。


In [ ]:
# define a new agent, this time with access to our tool!
agent_b = create_react_agent(
    model="openai:gpt-5",
    prompt="You are a helpful assistant that can help with everyday tasks."
           " If the user's request is confusing you must ask them to clarify"
           " their intent, and fulfill the instruction to the best of your"
           " ability. Be concise and friendly at all times."
           # It's useful to let the agent know about the tools it has at its disposal.
           " Use the Gmail tools that you have to address requests about emails.",
    tools=[gmail_tool], # we pass the tool we previously authorized.
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
        "user_id": os.getenv("ARCADE_USER_ID") # When using Arcade tools, we must provide the user_id on the LangGraph config, so Arcade can execute the tool invoked by the agent.
    }
}
print(f'thread_id = {config["configurable"]["thread_id"]}')

# we're using the same prompt we use before, but we're swapping the agent
prompt = "summarize my latest 3 emails please"
user_message = {"messages": [HumanMessage(content=prompt)]}
run_graph(agent_b, config, user_message)

# 多服务工具集成

在成功集成 Gmail 的基础上，我们现在扩展 Agent 的能力，纳入多个外部服务。本节演示如何在多个提供商之间高效管理认证，同时维持安全与用户体验的标准。

## 批量授权工具函数

随着 Agent 能力的扩展，逐个管理多个工具的授权会变得很繁琐。这需要为 Agent[初始化多个工具](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--multi-user-agent-arcade--multiuser-agent-arcade&click=arcade-docs&target=https%3A%2F%2Fdocs.arcade.dev%2Fhome%2Ffaq%23can-i-authenticate-multiple-tools-at-once%3Futm_source%3Dgithub%26utm_medium%3Dnotebook%26utm_campaign%3Dnir_diamant%26utm_content%3Dtutorial&text=initializing%20multiple%20tools)，并对每个工具的作用域进行认证。下面的函数通过按提供商对授权作用域进行分组来简化这一过程，最大程度减少用户需要完成的 OAuth 流程次数，同时确保全面的工具访问。


In [ ]:
def authorize_tools(tools, user_id, client):

    # This will map all the providers to the specific scopes they need
    provider_to_scopes = {}
    for tool in tools:
        provider = tool.requirements.authorization.provider_id
        if provider not in provider_to_scopes:
            provider_to_scopes[provider] = set()

        if tool.requirements.authorization.oauth2.scopes:
            provider_to_scopes[provider] |= set(tool.requirements.authorization.oauth2.scopes)

    # Each provider will handle its own scopes, we iterate and present the
    # auth URL for all providers that need it
    for provider, scopes in provider_to_scopes.items():
        # start auth
        auth_response = client.auth.start(
            user_id=user_id,
            scopes=list(scopes),
            provider=provider
        )

        # show the url to the user if needed
        if auth_response.status != "completed":
            print(f"🔗 Please click here to authorize: {auth_response.url}")
            print(f"⏳ Waiting for authorization completion...")

            # Wait for the authorization to complete with timeout
            client.auth.wait_for_completion(auth_response),


## 完整的工具套件配置

我们现在通过加入用于邮件发送、Slack 通信和 Notion 内容管理的工具来扩展 Agent 的能力。这种配置让 Agent 不仅能从各种服务读取信息，还能创建和发送内容，从而实现更复杂的流程自动化。


In [ ]:
# add a single tool
manager.add_tool("Gmail.SendEmail")
# add an entire toolkit (a collection of tools)
manager.add_toolkit("Slack")
manager.add_toolkit("NotionToolkit")

## 多服务授权

下面的单元格同时为所有已配置工具执行授权流程。这种高效的方式在建立 Gmail、Slack 和 Notion 访问所需权限的同时，最大程度减少用户交互。批量授权系统会自动按提供商对作用域进行分组，以呈现最少数量的授权流程。


In [ ]:
authorize_tools(
    tools=manager.definitions,
    user_id=os.getenv("ARCADE_USER_ID"),
    client=arcade_client
)

## 多服务 Agent 实现

在完成全面的工具授权后，我们创建了迄今能力最强的 Agent。该实现利用 ToolManager 的 LangChain 转换功能，在 Arcade 的工具定义与 LangGraph 的执行框架之间实现无缝集成。增强后的提示词引导 Agent 针对不同类型的请求选择合适的工具。


In [ ]:
# define a new agent, this time with access to our tool!
agent_c = create_react_agent(
    model="openai:gpt-5",
    prompt="You are a helpful assistant that can help with everyday tasks."
           " If the user's request is confusing you must ask them to clarify"
           " their intent, and fulfill the instruction to the best of your"
           " ability. Be concise and friendly at all times."
           # It's useful to let the agent know about the tools it has at its disposal.
           " Use the Gmail tools to address requests about reading or sending emails."
           " Use the Slack tools to address requests about interactions with users and channels in Slack."
           " Use the Notion tools to address requests about managing content in Notion Pages."
           " In general, when possible, use the most relevant tool for the job.",
    tools=manager.to_langchain(),
    checkpointer=checkpointer
)



## 复杂多服务任务执行

这个演示展示了我们的 Agent 跨多个服务编排复杂工作流的能力。该请求要求 Agent 分析邮件数据、获取 Slack 通信记录，并探索 Notion 工作区结构，展示了复杂的工具选择与执行协调能力。


In [ ]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
        "user_id": os.getenv("ARCADE_USER_ID") # When using Arcade tools, we must provide the user_id on the LangGraph config, so Arcade can execute the tool invoked by the agent.
    }
}
print(f'thread_id = {config["configurable"]["thread_id"]}')

# we're using the same prompt we use before, but we're swapping the agent
prompt = "summarize my latest 3 emails, then show me the latest 3 messages in the #general Slack channel, and tell me about the structure of my Notion Workspace"
user_message = {"messages": [HumanMessage(content=prompt)]}
run_graph(agent_c, config, user_message)

# 实现 Human-in-the-Loop 安全机制

虽然我们的多服务 Agent 展示了令人印象深刻的能力，但生产系统需要健壮的安全机制来防止意外操作。本节为敏感操作实现 human-in-the-loop 控制，确保潜在有害或不可逆的操作在执行前必须经过用户的明确批准。

## 识别敏感操作

在实现安全控制之前，我们必须先确定哪些工具需要人工监督。下面的工具检查有助于我们根据潜在影响和不可逆性对操作进行分类。


In [ ]:
for tool_name, _ in manager:
    print(tool_name)

## 敏感工具分类

基于潜在影响分析，我们识别出那些若以错误参数执行可能会造成意外后果的工具。这些工具通常涉及创建、发送或修改数据，而不仅仅是获取信息。该分类聚焦于具有外部影响，或可能危及用户隐私或系统完整性的操作。


In [ ]:
tools_to_protect = [
    "Gmail_SendEmail",
    "Slack_SendDmToUser",
    "Slack_SendMessage",
    "Slack_SendMessageToChannel",
    "NotionToolkit_AppendContentToEndOfPage",
    "NotionToolkit_CreatePage",
]

## Human-in-the-Loop 工具包装器

下面的实现创建了一个包装函数，把普通工具转换为受人工监督的版本。该包装器拦截工具执行请求，把计划执行的操作展示给用户审批，只有收到明确同意后才继续执行。该实现利用 LangGraph 的 interrupt 机制暂停执行，等待用户输入。


In [ ]:
from typing import Callable, Any
from langchain_core.tools import tool, BaseTool
from langgraph.types import interrupt, Command
from langchain_core.runnables import RunnableConfig
import pprint


def add_human_in_the_loop(
    target_tool: Callable | BaseTool,
) -> BaseTool:
    """Wrap a tool to support human-in-the-loop review."""
    if not isinstance(target_tool, BaseTool):
        target_tool = tool(target_tool)

    @tool(
        target_tool.name,
        description=target_tool.description,
        args_schema=target_tool.args_schema
    )
    def call_tool_with_interrupt(config: RunnableConfig, **tool_input):

        arguments = pprint.pformat(tool_input, indent=4)
        response = interrupt(
            f"Do you allow the call to {target_tool.name} with arguments:\n"
            f"{arguments}"
        )

        # approve the tool call
        if response == "yes":
            tool_response = target_tool.invoke(tool_input, config)
        # deny tool call
        elif response == "no":
            tool_response = "The User did not allow the tool to run"
        else:
            raise ValueError(
                f"Unsupported interrupt response type: {response}"
            )

        return tool_response

    return call_tool_with_interrupt


## 选择性工具保护应用

该实现有选择地应用 human-in-the-loop 保护，只包装被识别为敏感的工具，而保持只读操作不变。这种方法既能在安全操作上维持 Agent 效率，又能确保对潜在高风险操作进行适当监督。


In [ ]:
protected_tools = [
    add_human_in_the_loop(t)
    if t.name in tools_to_protect else t
    for t in manager.to_langchain()
]

## 中断处理工具函数

LangGraph 的 interrupt 需要专门的处理，以便在用户输入后恢复执行。下面的工具函数为审批决策提供了用户友好的接口，并自动化了使用用户响应恢复 Agent 执行的过程。yes/no 循环确保决策清晰，而中断处理器则管理执行恢复的技术细节。


In [ ]:
def yes_no_loop(prompt: str) -> str:
    """
    Force the user to say yes or no
    """
    print(prompt)
    user_input = input("Your response [y/n]: ")
    while user_input.lower() not in ["y", "n"]:
        user_input = input("Your response (must be 'y' or 'n'): ")
    return "yes" if user_input.lower() == "y" else "no"


def handle_interrupts(graph: CompiledStateGraph, config):
    for interr in graph.get_state(config).interrupts:
        approved = yes_no_loop(interr.value)
        run_graph(graph, config, Command(resume=approved))


## 受保护 Agent 的实现

我们最终的 Agent 实现整合了全面的安全控制，同时保留了本教程中开发的所有多服务能力。这个 Agent 代表了一个生产就绪的系统，在功能与安全之间取得平衡，确保用户对敏感操作保持控制，同时又能从常规任务的自动化辅助中受益。


In [ ]:
# define a new agent, this time with access to our tool!
agent_hitl = create_react_agent(
    model="openai:gpt-5",
    prompt="You are a helpful assistant that can help with everyday tasks."
           " If the user's request is confusing you must ask them to clarify"
           " their intent, and fulfill the instruction to the best of your"
           " ability. Be concise and friendly at all times."
           # It's useful to let the agent know about the tools it has at its disposal.
           " Use the Gmail tools to address requests about reading or sending emails."
           " Use the Slack tools to address requests about interactions with users and channels in Slack."
           " Use the Notion tools to address requests about managing content in Notion Pages."
           " In general, when possible, use the most relevant tool for the job.",
    tools=protected_tools,
    checkpointer=checkpointer
)

## 安全机制演示

下面的测试通过尝试发送一封可能敏感的邮件来演示我们的安全系统如何运作。该场景展示了 human-in-the-loop 机制如何拦截该操作、把细节呈现给用户审查，并在继续执行前等待明确批准。


In [ ]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4(),
        "user_id": os.getenv("ARCADE_USER_ID") # When using Arcade tools, we must provide the user_id on the LangGraph config, so Arcade can execute the tool invoked by the agent.
    }
}
print(f'thread_id = {config["configurable"]["thread_id"]}')

# we're using the same prompt we use before, but we're swapping the agent
prompt = 'send an email with subject "confidential data" and body "this is top secret information" to random-dude@example.com'
user_message = {"messages": [HumanMessage(content=prompt)]}
run_graph(agent_hitl, config, user_message)

## 中断状态检查

当我们的安全系统激活时，Agent 执行会暂停并进入中断状态。下面的检查会展示待处理的审批请求，演示系统如何捕获计划操作详情，并在继续之前等待用户决策。


In [ ]:
agent_hitl.get_state(config).interrupts

## 用户决策处理

下面的单元格处理待处理的中断，把操作详情呈现给用户并收集其审批决策。该演示展示了用户如何审查潜在敏感操作，并就是否继续执行 Agent 提议的操作做出明智的决定。


In [ ]:
handle_interrupts(agent_hitl, config)

## 完整的交互式系统

这个最终实现提供了一个完整的交互式系统，整合了本教程中开发的所有能力。用户可以自然地与一个能访问多个外部服务的 Agent 对话，同时通过 human-in-the-loop 控制保证敏感操作的安全。该系统在无缝的用户体验中自动处理授权、工具执行和安全审批。


In [ ]:
config = {
    "configurable": {
        "thread_id": uuid.uuid4()
    }
}
while True:
    user_input = input("👤: ")
    # let's use "exit" as a safe way to break the infinite loop
    if user_input.lower() == "exit":
        break

    user_message = {"messages": [HumanMessage(content=user_input)]}

    run_graph(agent_hitl, config, user_message)

    handle_interrupts(agent_hitl, config)